# measure acceleration

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## project:
- See if I can measure an acceleration in one of our precise *Kepler* clocks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt

In [ ]:
# all this to import the KeplerClocks code

from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks as kc

In [ ]:
# read in some useful clocks from a previous life

CLOCKS_DB_FILE = Path.cwd() / "../../KeplerClocks/data/clocks_2026-09-14.db"
conn = kc.get_db_connection(CLOCKS_DB_FILE)
query = "SELECT * FROM best_clock ORDER BY theoretical_value DESC;"
clocks = pd.read_sql_query(query, conn)
conn.close()
print(clocks[:5])

In [ ]:
# pull one clock and get the pars

ii = 4
clock = clocks.iloc[ii]
print(clock)
ts, ys, errs, df, dt = kc.get_kepler_data(clock["star_id"])
ivars = errs ** -2
X, ms, pars = kc.fourier_wls_fit(clock["angular_frequency"], clock["fourier_series_degree"],
                                 ts, ys, ivars)
print(ts.shape, X.shape, ms.shape, pars.shape)

In [ ]:
# get residuals and project onto the derivative wrt phase

resids = ys - X @ pars
deriv_pars = kc.take_derivative_wrt_phase(pars, ms)
derivs = clock['angular_frequency'] * X @ deriv_pars # dflux / dt
print(resids.shape, derivs.shape)

In [ ]:
# obtain an advance estimate in chunks

nchunk = 16
chunk_ts, chunk_advances, chunk_ivars = np.zeros(nchunk), np.zeros(nchunk), np.zeros(nchunk)
for chunk in range(nchunk):
    a, b = np.percentile(ts, [100 * chunk / nchunk, 100 * (chunk + 1) / nchunk])
    inchunk = (ts >= a) & (ts <= b)
    chunk_ts[chunk] = np.sum(ivars[inchunk] * ts[inchunk]) \
                    / np.sum(ivars[inchunk])
    chunk_advances[chunk] = np.sum(ivars[inchunk] * resids[inchunk] * derivs[inchunk]) \
                          / np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
    chunk_ivars[chunk]    = np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
print(chunk_advances)

In [ ]:
# is there a trend in the residuals?

f = plt.figure(figsize=(9, 3))
plt.axhline(0, c="k", lw=0.5)
plt.step(chunk_ts, chunk_advances, c="k", where="mid")
plt.errorbar(chunk_ts, chunk_advances, yerr = 10. / np.sqrt(chunk_ivars), marker="o", color="k", linestyle="none")
plt.xlabel("Barycentric time BJD [d]")
plt.ylabel("advance $O-C$ [s]")
plt.title(f"clock in {clock["star_id"]} with period {2. * np.pi / clock["angular_frequency"]:.5f} d")